# Leçon 13 - Mémoire d'Agent avec les Graphes de Connaissances Cognee


## Setup

This notebook demonstrates how to build an intelligent **coding assistant** with persistent memory using [**Cognee**](https://www.cognee.ai/) knowledge graphs and the **Microsoft Agent Framework** (MAF).

Cognee transforms unstructured text into a structured, queryable knowledge graph backed by vector embeddings — giving your agent a rich, relationship-aware long-term memory.

### What You'll Learn
1. **Build Knowledge Graphs**: Transform developer profiles and best practices into structured, queryable knowledge.
2. **Integrate Cognee with MAF**: Use `@tool` functions to let an MAF agent query Cognee's knowledge graph.
3. **Session-Aware Conversations**: Maintain context across multiple questions in the same session.
4. **Long-Term Memory**: Persist important knowledge across sessions and retrieve it in new conversations.

### Prerequisites
- Python 3.9+
- Redis running locally (`docker run -d -p 6379:6379 redis`) for session management
- An LLM API key (e.g. OpenAI) — set `LLM_API_KEY` in `.env`
- `CACHING=true` in `.env` (required for Cognee sessions)
- A Microsoft Foundry project with a deployed chat model
- `AZURE_AI_PROJECT_ENDPOINT` and `AZURE_AI_MODEL_DEPLOYMENT_NAME` in `.env`
- Azure CLI authenticated (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")


In [ ]:
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ FoundryChatClient created")


## Types de mémoire des agents

Ce carnet explore les mêmes trois types de mémoire que dans le carnet principal de la Leçon 13, mais utilise Cognee comme backend de mémoire à long terme :

| Type de mémoire | Mécanisme | Durée de vie |
|---|---|---|
| **Travail** | `agent.create_session()` (MAF) | Conversation unique |
| **Court terme** | Cache de session Cognee (Redis) | Session unique |
| **Long terme** | Graphe de connaissances Cognee + vecteurs | Permanent |

### Architecture de la mémoire de Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Préparer le stockage Cognee


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Partie 1 — Construire la base de connaissances

Nous intégrons trois types de données pour créer une base de connaissances complète pour notre assistant de codage :

1. **Profil du développeur** — expertise personnelle et parcours technique
2. **Meilleures pratiques Python** — le Zen de Python avec des directives pratiques
3. **Conversations historiques** — sessions de questions/réponses passées entre développeurs et assistants IA


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Visualiser le Graphe de Connaissances

Cognee peut afficher une visualisation HTML interactive des entités et des relations qu’il a extraites.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Enrichir la Mémoire avec Memify

`memify()` analyse le graphe de connaissances et génère des règles intelligentes — identifiant des motifs, des meilleures pratiques et des relations entre les concepts.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Partie 2 — Agent MAF avec les outils Cognee

Nous allons maintenant créer un agent MAF capable d’interroger le graphe de connaissances de Cognee via des fonctions `@tool`. Cela permet à l’agent de tirer pleinement parti de la recherche sémantique sensible au graphe tout en maintenant le contexte conversationnel à travers les sessions.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = provider.as_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")


## Mémoire de travail avec les sessions

Le `AgentSession` (créé via `agent.create_session()`) fournit une mémoire de travail au sein d'une session. L'agent peut se référer aux messages précédents tout en interrogeant le graphe de connaissances à long terme de Cognee.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Nouvelle session — La mémoire à long terme persiste

Commencer une nouvelle session efface la mémoire de travail, mais le graphe de connaissances Cognee est toujours disponible. L'agent peut récupérer les mêmes connaissances à long terme dans une conversation complètement nouvelle.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Résumé

Dans ce carnet, vous avez construit un assistant de codage qui combine la **mémoire de travail de MAF** (`agent.create_session()`) avec **le graphe de connaissances à long terme de Cognee**.

### Ce que vous avez appris
1. **Construction de graphe de connaissances** : Cognee ingère du texte non structuré et construit un graphe + une mémoire vectorielle.
2. **Enrichissement de graphe avec memify** : Faits dérivés et relations plus riches au-dessus de votre graphe existant.
3. **Intégration MAF + Cognee** : Les fonctions `@tool` permettent à un agent MAF d'interroger naturellement le graphe de Cognee.
4. **Mémoire de travail + mémoire à long terme** : `AgentSession` (via `agent.create_session()`) fournit le contexte de session tandis que Cognee offre une connaissance persistante.
5. **Recherche filtrée avec NodeSets** : Ciblez des sous-ensembles spécifiques du graphe de connaissances (par ex. uniquement les principes).

### Points clés à retenir
- **Cognee** transforme le texte brut en une mémoire structurée et consciente des relations — plus puissante qu'un simple magasin vectoriel plat.
- Les **fonctions `@tool`** font le lien proprement entre les agents MAF et les systèmes de connaissances externes.
- **`AgentSession`** (via `agent.create_session()`) maintient le contexte par conversation séparé des connaissances à long terme.
- Le même graphe de connaissances sert plusieurs sessions et agents.

### Applications dans le monde réel
- **Co-pilotes développeurs** : revue de code, analyse d'incidents, assistants d'architecture
- **Co-pilotes orientés client** : agents de support sur docs produits, FAQ et notes CRM
- **Co-pilotes experts internes** : assistants politiques, juridiques ou sécurité raisonnant sur des directives
- **Couches de données unifiées** : combiner données structurées et non structurées en un graphe interrogeable

### Étapes suivantes
- Expérimenter la conscience temporelle dans Cognee
- Définir une ontologie OWL pour la qualité de graphe spécifique au domaine
- Ajouter des boucles de rétroaction utilisateur pour améliorer la récupération au fil du temps
- Passer à des systèmes multi-agents partageant la même couche mémoire Cognee


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :
Ce document a été traduit à l'aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforçions d'assurer l'exactitude, veuillez noter que les traductions automatisées peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue native doit être considéré comme la source faisant autorité. Pour les informations critiques, il est recommandé de recourir à une traduction professionnelle réalisée par un humain. Nous ne saurions être tenus responsables des malentendus ou erreurs d'interprétation découlant de l'utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
